<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c41-header.png" alt="Nextia Learning · Speech AI: Transcribe, Understand and Speak" width="100%">

# Solution: Speaker embeddings and clustering

**[Speaker embeddings and clustering](https://learning.nextia-ai.com/courses/speech/m05/speaker-embeddings-and-clustering/)** · Speech AI: Transcribe, Understand and Speak · Module 5, lesson 2 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** turn short pieces of speech into speaker embeddings, compare them with the cosine similarity, group them into two speakers by clustering, and measure the result with the diarization error rate (DER) on mono calls.

| | |
|---|---|
| **Time** | About 40 minutes |
| **Needs** | An internet connection for the setup cells (a 27 MB speaker model is downloaded once). No account and no key. |
| **You should know** | The DER and the energy VAD, from [Split by channel first](https://learning.nextia-ai.com/courses/speech/m05/split-by-channel-first/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, SciPy 1.18.1, soundfile 0.14.0, matplotlib 3.11.2, scikit-learn 1.9.1 and sherpa-onnx 2.0.1, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/speech/m05/speaker-embeddings-and-clustering/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/M05-L02-speaker-embeddings-and-clustering/speaker-embeddings-and-clustering-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, scipy 1.18.1, soundfile 0.14.0, scikit-learn 1.9.1, sherpa-onnx 2.0.1, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 scipy==1.18.1 soundfile==0.14.0 matplotlib==3.11.2 scikit-learn==1.9.1 sherpa-onnx==2.0.1
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'scipy', 'soundfile', 'scikit-learn', 'sherpa-onnx']:
    print(name, version(name))

## Setup: get the Larkfield calls

Run the next cell. It downloads the files that this lesson needs (about 4 MB) into a folder `larkfield-calls`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-calls` (in Colab: the **Files** panel on the left), then run the cell again.

Where the calls come from: Larkfield is an invented shop. Our code wrote the scripts, and open text-to-speech voices (Piper and Kokoro) read them. Every call is **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/data/dataset-card.md) names each voice and its licence. The calls are free to reuse under CC BY 4.0.

In [ ]:
# Setup: download the files into larkfield-calls/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/"
FILES = {
    "data/calls/calls.jsonl": "22fe4e147bf62b4ff6c3b6b4a1cc559b048e12a0c2b3c8098eb72982f2d9e695",
    "data/calls/phone/C-0052.wav": "5d7b9d117a69ca8bff57a6ca7d7bb18ae5548d2b01fcd8c5a67cb1beb6a042ba",
    "data/calls/phone/C-0285.wav": "a6257abfda3e7d94171294ac23def33ab60a31e78f2cfdd1f7bffdf544eb68db",
    "data/calls/phone/C-0144.wav": "4d0e80b1dba9a9b12372ca31e047550fe50ef6df4b62b8fc41bd5bd0561976ce",
    "data/calls/phone/C-0217.wav": "ee4b12fdb7a34aabc34f4bef71c04f3dd523c8d06f61b0c217240f5ae94fee7e",
    "compare.py": "db394eb044e0423706644ba7c8e6451e4d21a268293b868032918ea2f30db85b",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-calls").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-calls and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-calls.")

## Setup: helpers

The next cell loads NumPy as `np`, soundfile as `sf`, SciPy's `signal`, matplotlib, the audio player `Audio`, and the course helper as `cp` (normalisation, word error rate and the comparison table). It defines a few small helpers:

- `load_calls()` reads the labels of every call; `reference(call)` joins its turns into the reference transcript.
- `read_audio(path, sr, channel)` reads a recording as numbers between -1 and 1; `dbfs(x)` gives its loudness.
- `true_turns(call)` gives the true (start, end, speaker) of each turn, from the first word to the last word.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: calls, audio, and checks that do not show the answer.
import hashlib, json, warnings
from math import gcd
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import Audio, display
import compare as cp
warnings.filterwarnings("ignore")

def load_calls(path="data/calls/calls.jsonl"):
    """The labels of every call, by call ID."""
    rows = [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]
    return {c["call_id"]: c for c in rows}

def reference(call, speaker=None):
    """The reference transcript: the text of all turns (or of one speaker's turns), in time order."""
    return " ".join(t["text"] for t in call["turns"] if speaker in (None, t["speaker"]))

def read_audio(path, sr=None, channel=None):
    """(numbers between -1 and 1, samples a second). channel=None mixes the channels; 0 or 1 keeps one side."""
    x, file_sr = sf.read(str(path), dtype="float32", always_2d=True)
    x = x.mean(axis=1) if channel is None else x[:, channel]
    if sr and sr != file_sr:
        g = gcd(file_sr, sr)
        x = signal.resample_poly(x, sr // g, file_sr // g).astype(np.float32)
    return x, sr or file_sr

def dbfs(x):
    """Loudness in decibels below full scale: 0 is the loudest; speech on a phone is often -30 to -15."""
    return float(20 * np.log10(np.sqrt(np.mean(np.square(x))) + 1e-9))

def true_turns(call):
    """The true turns: (start, end, speaker), from the first word to the last word of each turn."""
    return [(t["words"][0]["start"], t["words"][-1]["end"], t["speaker"]) if t.get("words")
            else (t["start"], t["end"], t["speaker"]) for t in call["turns"]]

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The tools of the last lesson

Run the next two cells. They define the energy VAD of Module 3 and `der`, the diarization error rate of the last lesson.

In [ ]:
FRAME = 0.03  # 30 ms frames

def frame_db(x, sr):
    """The loudness of each 30 ms frame, in dBFS."""
    n = int(sr * FRAME)
    frames = x[: len(x) // n * n].reshape(-1, n)
    return 20 * np.log10(np.sqrt(np.mean(frames ** 2, axis=1)) + 1e-9)

def regions(flags, hangover=0.3, min_speech=0.1):
    """Speech frames -> (start, end) times. A pause shorter than `hangover` seconds stays inside the speech."""
    out, start, quiet = [], None, 0
    keep = int(round(hangover / FRAME))
    for i, f in enumerate(flags):
        if f:
            start = i if start is None else start
            quiet = 0
        elif start is not None:
            quiet += 1
            if quiet > keep:
                end = i - quiet + 1
                if (end - start) * FRAME >= min_speech:
                    out.append((start * FRAME, end * FRAME))
                start, quiet = None, 0
    if start is not None:
        out.append((start * FRAME, (len(flags) - quiet) * FRAME))
    return out

def energy_vad(x, sr, threshold_db=-40.0, hangover=0.3):
    """The baseline: a frame is speech when it is louder than threshold_db."""
    return regions(frame_db(x, sr) > threshold_db, hangover)

In [ ]:
import itertools
STEP = 0.01

def der(pred, truth, seconds, collar=0.25):
    """Diarization error rate = (missed speech + false speech + speaker confusion) / true speech time.
    Turns are (start, end, speaker). Speaker names are matched in the best way first.
    A collar of 0.25 s around each true boundary is not scored."""
    n = int(seconds / STEP) + 1
    tnames, pnames = sorted({t[2] for t in truth}), sorted({p[2] for p in pred})
    def masks(turns, names):
        m = np.zeros((len(names), n), bool)
        for s, e, w in turns:
            m[names.index(w), int(s / STEP):int(e / STEP)] = True
        return m
    T, P = masks(truth, tnames), masks(pred, pnames)
    score = np.ones(n, bool)
    for s, e, _ in truth:
        for b in (s, e):
            score[max(0, int((b - collar) / STEP)):int((b + collar) / STEP)] = False
    best = None
    for perm in itertools.permutations(range(max(len(pnames), len(tnames))), len(pnames)):
        conf = 0
        for pi, ti in enumerate(perm):
            other = (P[pi] & ~T[ti]) if ti < len(tnames) else P[pi]
            conf += int((other & T.any(0) & score).sum())
        best = conf if best is None else min(best, conf)
    missed = int((T.any(0) & ~P.any(0) & score).sum())
    false = int((P.any(0) & ~T.any(0) & score).sum())
    total = max(int((T.any(0) & score).sum()), 1)
    return {"der": (missed + false + best) / total, "missed": missed / total, "false": false / total,
            "confusion": best / total}

## 2. The speaker model

A **speaker embedding** is a list of numbers that describes how a voice sounds, not what it says. A speaker model makes it. Two pieces of speech by the same person give lists that point in nearly the same direction.

The next two cells download the project's speaker model once (27 MB, checked against its SHA-256), and defines `embed(x)`: one piece of audio in, one embedding out.

In [ ]:
import hashlib, subprocess, urllib.request
from pathlib import Path

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

In [ ]:
# The speaker model: WeSpeaker ResNet-34, trained on VoxCeleb (Apache-2.0), the sherpa-onnx copy (27 MB).
import sherpa_onnx
# Two places for the same file (the checksum is checked either way): GitHub, then a copy on Hugging Face.
MODEL_URLS = ["https://github.com/k2-fsa/sherpa-onnx/releases/download/speaker-recongition-models/"
              "wespeaker_en_voxceleb_resnet34_LM.onnx",
              "https://huggingface.co/csukuangfj/speaker-embedding-models/resolve/main/"
              "wespeaker_en_voxceleb_resnet34_LM.onnx"]
MODEL_SHA256 = "e9848563da86f263117134dfd7ad63c92355b37de492b55e325400c9d9c39012"
model_file = Path("models/wespeaker_en_voxceleb_resnet34_LM.onnx")
if not (model_file.exists() and sha256(model_file) == MODEL_SHA256):
    model_file.parent.mkdir(exist_ok=True)
    for url in MODEL_URLS:
        try:
            download(url, model_file)
        except SystemExit:
            continue  # this server did not answer: try the next one
        if model_file.exists() and sha256(model_file) == MODEL_SHA256:
            break
    else:
        raise SystemExit("Could not download the speaker model with the right checksum. Delete the folder models "
                         "and run again later.")
extractor = sherpa_onnx.SpeakerEmbeddingExtractor(
    sherpa_onnx.SpeakerEmbeddingExtractorConfig(model=str(model_file), num_threads=2))

def embed(x, sr=16000):
    """One piece of speech -> one speaker embedding, scaled to length 1."""
    stream = extractor.create_stream()
    stream.accept_waveform(sr, x)
    stream.input_finished()
    e = np.array(extractor.compute(stream))
    return e / np.linalg.norm(e)

print("Speaker model ready: each embedding has", extractor.dim, "numbers.")

> **Check.** Each embedding has 256 numbers.

## 3. The cosine similarity on a tiny example

We compare two embeddings with the **cosine similarity**: 1 when they point the same way, 0 when they are unrelated, -1 when they point opposite ways. For lists of length 1, it is the the sum of the products of the numbers.

> **Predict.** `a` = (0.6, 0.8), `b` = (0.8, 0.6), `c` = (-0.6, 0.8). Which pair is more alike?

In [ ]:
a = np.array([0.6, 0.8])      # three tiny "voices" with two numbers each
b = np.array([0.8, 0.6])
c = np.array([-0.6, 0.8])

def cosine(u, v):
    """1 = the same direction, 0 = unrelated, -1 = opposite."""
    return float(u @ v / (np.linalg.norm(u) * np.linalg.norm(v)))

print("a, b:", round(cosine(a, b), 2), "  a, c:", round(cosine(a, c), 2))

> **Check.** a and b: 0.6 × 0.8 + 0.8 × 0.6 = 0.96. a and c: 0.6 × (-0.6) + 0.8 × 0.8 = 0.28. a and b are more alike.

## 4. Embed three turns of a call

Call C-0052 is stereo, but we mix it into one channel, as a mono recording would. We embed two turns of the agent and one turn of the caller, and compare all three.

> **Predict.** Is the cosine of the two agent turns higher than the cosine of an agent turn and the caller turn?

In [ ]:
calls = load_calls()
call = calls["C-0052"]
x, sr = read_audio("data/calls/phone/C-0052.wav", 16000)       # both sides mixed, as in a mono call
truth = true_turns(call)
chosen = [0, 2, 3]                                             # agent, agent, caller
vectors = []
for i in chosen:
    s, e, who = truth[i]
    vectors.append(embed(x[int(s * sr):int(e * sr)], sr))
    print(i, who, f"{s:.1f}-{e:.1f} s", call["turns"][i]["text"][:50])
sims = np.array([[float(u @ v) for v in vectors] for u in vectors])
print(np.round(sims, 3))

> **Check.** The two agent turns: 0.916. The agent and the caller: 0.767 and 0.809. The same voice is closer, but not 1: the words and the line change the numbers a little.

## 5. Cut a mono call into pieces

Call C-0285 is a mono call. We do not know where the turns are, so we cut every speech region into **pieces** of 1.5 s, a new piece every 0.75 s, and embed each piece. A leftover piece shorter than 1 s is dropped: one odd, short piece can otherwise take a whole cluster for itself.

In [ ]:
mono_call = calls["C-0285"]
m, msr = read_audio("data/calls/phone/C-0285.wav", 16000)

def pieces_of(x, sr, window=1.5, hop=0.75, threshold_db=-40, shortest=1.0):
    """Speech regions -> pieces of `window` seconds, a new piece every `hop` seconds.
    The last piece of a region can be shorter; pieces under `shortest` seconds are dropped."""
    out = []
    for s, e in energy_vad(x, sr, threshold_db):
        t = s
        while t < e - 0.4:
            if min(t + window, e) - t >= min(shortest, window):
                out.append((t, min(t + window, e)))
            t += hop
    return out

pieces = pieces_of(m, msr)
embs = np.stack([embed(m[int(s * msr):int(e * msr)], msr) for s, e in pieces])
print(len(pieces), "pieces;", embs.shape)

## 6. Cluster the pieces into two speakers

**Clustering** puts similar things in the same group, without labels. Agglomerative clustering starts with every piece in its own group, then joins the two closest groups again and again, until two groups are left. "Closest" is the cosine distance (1 minus the cosine similarity), averaged between the groups.

The group of the first piece is named "agent", because Larkfield's agents speak first. Then each piece gives its time to its speaker.

In [ ]:
from sklearn.cluster import AgglomerativeClustering

def cluster_turns(pieces, embs, n_speakers=2):
    """Two groups of similar embeddings; the group of the first piece is the agent (agents answer the call)."""
    labels = AgglomerativeClustering(n_clusters=n_speakers, metric="cosine", linkage="average").fit(embs).labels_
    names = {labels[0]: "agent"}
    turns = []
    for (s, e), lab in zip(pieces, labels):
        who = names.setdefault(lab, "caller")
        if turns and turns[-1][2] == who and s <= turns[-1][1]:
            turns[-1] = (turns[-1][0], e, who)          # join overlapping pieces of one speaker
        else:
            if turns and s < turns[-1][1]:
                s = turns[-1][1]                        # pieces overlap: the new speaker starts where the last ends
            turns.append((s, e, who))
    return turns, labels

pred, labels = cluster_turns(pieces, embs)
mono_truth = true_turns(mono_call)
result = der(pred, mono_truth, mono_call["duration"])
print({k: round(v, 3) for k, v in result.items()})

> **Check.** 34 pieces. DER 0.189: missed 0.169, false 0.0, confusion 0.02. Confusion is now possible, because one channel holds both voices.

The next cell draws every piece's embedding in two dimensions (the two directions in which the embeddings differ most, found by PCA). The colour is the cluster; the shape is the true speaker.

In [ ]:
centred = embs - embs.mean(axis=0)                    # PCA by hand: the two directions of most spread
_, _, vt = np.linalg.svd(centred, full_matrices=False)
xy = centred @ vt[:2].T

def true_speaker(s, e):
    overlap = {}
    for a, b, who in mono_truth:
        overlap[who] = overlap.get(who, 0) + max(0, min(e, b) - max(s, a))
    return max(overlap, key=overlap.get)

shapes = {"agent": "o", "caller": "s"}
fig, ax = plt.subplots(figsize=(6, 5))
for (s, e), (px, py), lab in zip(pieces, xy, labels):
    ax.scatter(px, py, marker=shapes[true_speaker(s, e)], color="tab:blue" if lab == labels[0] else "tab:orange")
ax.set_title("C-0285: one dot per piece; colour = cluster, circle = agent, square = caller (truth)")
plt.show()

## 7. Change one thing: shorter pieces

> **Predict.** With 0.75 s pieces instead of 1.5 s, each piece is less likely to hold two voices, but it has less sound to describe the voice. Does the confusion go up or down?

In [ ]:
for window in (1.5, 0.75):
    p = pieces_of(m, msr, window=window, hop=window / 2)
    e = np.stack([embed(m[int(s * msr):int(t * msr)], msr) for s, t in p])
    r = der(cluster_turns(p, e)[0], mono_truth, mono_call["duration"])
    print(window, len(p), {k: round(v, 3) for k, v in r.items()})

> **Check.** 1.5 s pieces: DER 0.189 (confusion 0.02). 0.75 s pieces: DER 0.404 (confusion 0.184). The project keeps 1.5 s. Try other values on several calls before you change it: one call is not a test set.

## 8. Failure case: two voices that sound alike

Call C-0144 is a practice problem we made on purpose: the caller's voice was chosen to sound like the agent's.

In [ ]:
sim_call = calls["C-0144"]
y, ysr = read_audio("data/calls/phone/C-0144.wav", 16000)
sim_truth = true_turns(sim_call)
agent_v = np.mean([embed(y[int(s * ysr):int(e * ysr)], ysr) for s, e, w in sim_truth if w == "agent"], axis=0)
caller_v = np.mean([embed(y[int(s * ysr):int(e * ysr)], ysr) for s, e, w in sim_truth if w == "caller"], axis=0)
print("agent and caller voices, cosine:", round(cosine(agent_v, caller_v), 3))
yp = pieces_of(y, ysr)
ye = np.stack([embed(y[int(s * ysr):int(e * ysr)], ysr) for s, e in yp])
r = der(cluster_turns(yp, ye)[0], sim_truth, sim_call["duration"])
print({k: round(v, 3) for k, v in r.items()})

> **Check.** The two voices have a cosine of 0.959 in this call. DER 0.518, confusion 0.494. When two voices are this close, clustering cannot keep them apart. **Fix:** ask the phone system for stereo, or check the speaker names of important sentences (a promise) by another way: the next lessons.

## 9. Your turn: another mono call

> **Your turn.** Run the same pipeline on call C-0217 (`pieces_of`, `embed`, `cluster_turns`), compute its DER against `true_turns(task_call)`, and put the confusion part, rounded to 3 decimals, in `task_confusion`.

In [ ]:
task_call = calls["C-0217"]
z, zsr = read_audio("data/calls/phone/C-0217.wav", 16000)
zp = pieces_of(z, zsr)
ze = np.stack([embed(z[int(s * zsr):int(e * zsr)], zsr) for s, e in zp])
task_result = der(cluster_turns(zp, ze)[0], true_turns(task_call), task_call["duration"])
task_confusion = round(task_result["confusion"], 3)
print(task_confusion)

In [ ]:
expect('task_confusion', task_confusion, 0.125, tolerance=0.002)

## Next

You turned pieces of speech into embeddings, clustered them into two speakers and measured the DER on mono calls. The [lesson page](https://learning.nextia-ai.com/courses/speech/m05/speaker-embeddings-and-clustering/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Pretrained diarization pipelines](https://learning.nextia-ai.com/courses/speech/m05/pretrained-diarization-pipelines/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/speech/m05/speaker-embeddings-and-clustering/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). Every call is synthetic: open text-to-speech voices read scripts that we invented. No real customer and no real person's voice is in this data.